# Signal Preprocessing and Window Specification

This notebook is the next implementation step after the ML-readiness audit.

It defines the common input space for Voisard and Felius, creates participant-level development folds, and verifies raw-signal loading on representative trials.

It does not train a classifier and does not overwrite the original EDA notebook.


## Processing contract

Initial common input:

- lower-back IMU;
- left-foot IMU;
- right-foot IMU;
- acceleration and gyroscope channels;
- acceleration in g;
- gyroscope in degrees per second;
- target sampling rate of 100 Hz;
- fixed 5-second windows;
- participant-level splitting before window generation.

Voisard walking bounds come from its annotated gait events. Felius has no equivalent event annotations in the local release, so its first pass is flagged as full-trial segmentation and requires a later walking-activity validation.


In [1]:
import json
import math
import sys
from fractions import Fraction
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.signal import resample_poly
from sklearn.model_selection import StratifiedGroupKFold

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

MANIFEST_PATH = PROJECT_ROOT / "data" / "interim" / "ml_readiness_manifest.csv"
manifest = pd.read_csv(MANIFEST_PATH)
manifest["participant_key"] = manifest["dataset_id"] + ":" + manifest["subject"].astype(str)

TARGET_FS_HZ = 100.0
WINDOW_SECONDS = 5.0
WINDOW_SAMPLES = int(TARGET_FS_HZ * WINDOW_SECONDS)
TRAIN_HOP_SAMPLES = int(WINDOW_SAMPLES * 0.50)
EVAL_HOP_SAMPLES = WINDOW_SAMPLES

print("Project root:", PROJECT_ROOT)
print("Manifest rows:", len(manifest))
print("Window samples:", WINDOW_SAMPLES)


Project root: C:\Users\frank\Documents\MR-ICT Review Paper
Manifest rows: 865
Window samples: 500


In [2]:
participant_table = (
    manifest.groupby(["participant_key", "dataset_id", "dataset", "subject"], as_index=False)
    .agg(
        label=("label", "first"),
        n_trials=("trial_id", "nunique"),
    )
)

participant_table["label_binary"] = participant_table["label"].map({"healthy": 0, "stroke": 1})
participant_table.head()


,participant_key,dataset_id,dataset,subject,label,n_trials,label_binary
0,felius_2024:H1,felius_2024,Felius,H1,healthy,2,0
1,felius_2024:H10,felius_2024,Felius,H10,healthy,1,0
2,felius_2024:H11,felius_2024,Felius,H11,healthy,2,0
3,felius_2024:H12,felius_2024,Felius,H12,healthy,1,0
4,felius_2024:H13,felius_2024,Felius,H13,healthy,2,0


In [3]:
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
split_rows = []

for fold_id, (train_idx, validation_idx) in enumerate(
    splitter.split(
        participant_table,
        participant_table["label_binary"],
        groups=participant_table["participant_key"],
    )
):
    train_keys = set(participant_table.iloc[train_idx]["participant_key"])
    validation_keys = set(participant_table.iloc[validation_idx]["participant_key"])

    for key in participant_table["participant_key"]:
        split_rows.append({
            "participant_key": key,
            "fold": fold_id,
            "role": "training" if key in train_keys else "validation",
        })

participant_splits = pd.DataFrame(split_rows)
split_audit = participant_splits.merge(
    participant_table[["participant_key", "dataset", "label"]],
    on="participant_key",
    how="left",
)

print("Participant-level fold counts:")
print(split_audit.groupby(["fold", "role", "label"]).size().rename("participants"))
print()
print("Each participant appears once per fold:")
print(split_audit.groupby(["participant_key", "fold"]).size().value_counts().sort_index())


Participant-level fold counts:
fold  role        label  
0     training    healthy     85
                  stroke     145
      validation  healthy     22
                  stroke      36
1     training    healthy     85
                  stroke     145
      validation  healthy     22
                  stroke      36
2     training    healthy     86
                  stroke     144
      validation  healthy     21
                  stroke      37
3     training    healthy     86
                  stroke     145
      validation  healthy     21
                  stroke      36
4     training    healthy     86
                  stroke     145
      validation  healthy     21
                  stroke      36
Name: participants, dtype: int64

Each participant appears once per fold:
1    1440
Name: count, dtype: int64


In [4]:
def cross_dataset_split_table(manifest_table):
    rows = []
    for held_out_dataset in ["voisard_2025", "felius_2024"]:
        direction = f"train_other_test_{held_out_dataset}"
        for _, row in manifest_table[["participant_key", "dataset_id", "label"]].drop_duplicates().iterrows():
            rows.append({
                "participant_key": row["participant_key"],
                "protocol": direction,
                "role": "test" if row["dataset_id"] == held_out_dataset else "development",
                "label": row["label"],
            })
    return pd.DataFrame(rows)


cross_dataset_splits = cross_dataset_split_table(
    manifest[["participant_key", "dataset_id", "label"]]
)

print("Cross-dataset participant counts:")
print(cross_dataset_splits.groupby(["protocol", "role", "label"]).size().rename("participants"))


Cross-dataset participant counts:
protocol                       role         label  
train_other_test_felius_2024   development  healthy     73
                                            stroke      49
                               test         healthy     34
                                            stroke     132
train_other_test_voisard_2025  development  healthy     34
                                            stroke     132
                               test         healthy     73
                                            stroke      49
Name: participants, dtype: int64


In [5]:
CHANNEL_ORDER = [
    "LB_acc_x", "LB_acc_y", "LB_acc_z", "LB_gyr_x", "LB_gyr_y", "LB_gyr_z",
    "LF_acc_x", "LF_acc_y", "LF_acc_z", "LF_gyr_x", "LF_gyr_y", "LF_gyr_z",
    "RF_acc_x", "RF_acc_y", "RF_acc_z", "RF_gyr_x", "RF_gyr_y", "RF_gyr_z",
]

ACCELERATION_MS2_TO_G = 1.0 / 9.80665
RAD_TO_DEG = 180.0 / np.pi


def resample_signal(signal, source_fs, target_fs=TARGET_FS_HZ):
    if float(source_fs) == float(target_fs):
        return signal
    ratio = Fraction(float(target_fs) / float(source_fs)).limit_denominator(1000)
    return resample_poly(signal, ratio.numerator, ratio.denominator, axis=0)


def walking_bounds_from_voisard(meta):
    uturn_start, uturn_end = meta["uturnBoundaries"]
    segments = []
    events = []
    for event_name in ["leftGaitEvents", "rightGaitEvents"]:
        events.extend(meta.get(event_name) or [])

    pre = [event for event in events if event[1] < uturn_start]
    post = [event for event in events if event[0] > uturn_end]

    if pre:
        segments.append((min(event[0] for event in pre), max(event[1] for event in pre)))
    if post:
        segments.append((min(event[0] for event in post), max(event[1] for event in post)))

    if not segments:
        return None
    return sorted(set(segments))


def load_voisard_trial(row):
    trial_dir = PROJECT_ROOT / row["trial_directory"]
    trial_id = row["trial_id"]
    meta_path = trial_dir / f"{trial_id}_meta.json"
    meta = json.loads(meta_path.read_text(encoding="utf-8"))

    frames = []
    for sensor in ["LB", "LF", "RF"]:
        path = trial_dir / f"{trial_id}_raw_data_{sensor}.txt"
        frame = pd.read_csv(path, sep="\t")
        frame = frame[["Acc_X", "Acc_Y", "Acc_Z", "Gyr_X", "Gyr_Y", "Gyr_Z"]].astype(float)
        values = frame.to_numpy()
        values[:, :3] *= ACCELERATION_MS2_TO_G
        values[:, 3:] *= RAD_TO_DEG
        frames.append(values)

    lengths = [len(frame) for frame in frames]
    n_samples = min(lengths)
    signal = np.concatenate([frame[:n_samples] for frame in frames], axis=1)
    return signal, float(meta["freq"]), walking_bounds_from_voisard(meta), "voisard_event_bounds"


def load_felius_trial(row):
    files = row["raw_files"].split("|")
    paths = {Path(path).name.rsplit("_", 1)[-1].replace(".csv", ""): PROJECT_ROOT / path for path in files}

    frames = []
    for sensor_key in ["lowback", "leftfoot", "rightfoot"]:
        frame = pd.read_csv(paths[sensor_key])
        frame = frame[["ax", "ay", "az", "gx", "gy", "gz"]].astype(float)
        frames.append(frame.to_numpy())

    lengths = [len(frame) for frame in frames]
    n_samples = min(lengths)
    signal = np.concatenate([frame[:n_samples] for frame in frames], axis=1)
    return signal, 100.0, [(0, n_samples)], "felius_full_trial_no_event_annotation"


def load_common_trial(row):
    if row["dataset_id"] == "voisard_2025":
        signal, source_fs, bounds, method = load_voisard_trial(row)
    elif row["dataset_id"] == "felius_2024":
        signal, source_fs, bounds, method = load_felius_trial(row)
    else:
        raise ValueError(f"Unsupported dataset: {row['dataset_id']}")

    signal = resample_signal(signal, source_fs, TARGET_FS_HZ)
    return signal, bounds, method


In [6]:
voisard_sample = manifest[manifest["dataset_id"] == "voisard_2025"].iloc[0]
felius_sample = manifest[manifest["dataset_id"] == "felius_2024"].iloc[0]

voisard_signal, voisard_bounds, voisard_method = load_common_trial(voisard_sample)
felius_signal, felius_bounds, felius_method = load_common_trial(felius_sample)

print("Voisard sample:", voisard_signal.shape, voisard_method, voisard_bounds)
print("Felius sample:", felius_signal.shape, felius_method, felius_bounds)
print("Expected channel count:", len(CHANNEL_ORDER))
assert voisard_signal.shape[1] == len(CHANNEL_ORDER)
assert felius_signal.shape[1] == len(CHANNEL_ORDER)
assert np.isfinite(voisard_signal).all()
assert np.isfinite(felius_signal).all()


Voisard sample: (1987, 18) voisard_event_bounds [(431, 1047), (1229, 1828)]
Felius sample: (13217, 18) felius_full_trial_no_event_annotation [(0, 13217)]
Expected channel count: 18


In [7]:
def enumerate_windows(n_samples, bounds, hop_samples):
    windows = []
    for start, end in bounds:
        end = min(int(end), n_samples)
        start = max(int(start), 0)
        if end - start < WINDOW_SAMPLES:
            continue
        for window_start in range(start, end - WINDOW_SAMPLES + 1, hop_samples):
            windows.append({
                "start_sample": window_start,
                "end_sample": window_start + WINDOW_SAMPLES,
            })
    return windows


def build_trial_window_spec(row, signal, bounds):
    train_windows = enumerate_windows(len(signal), bounds, TRAIN_HOP_SAMPLES)
    eval_windows = enumerate_windows(len(signal), bounds, EVAL_HOP_SAMPLES)
    return {
        "dataset_id": row["dataset_id"],
        "participant_key": row["participant_key"],
        "trial_id": row["trial_id"],
        "label": row["label"],
        "n_samples": len(signal),
        "segment_method": "event_bounds" if row["dataset_id"] == "voisard_2025" else "full_trial_no_event_annotation",
        "train_window_count": len(train_windows),
        "eval_window_count": len(eval_windows),
    }


sample_specs = pd.DataFrame([
    build_trial_window_spec(voisard_sample, voisard_signal, voisard_bounds),
    build_trial_window_spec(felius_sample, felius_signal, felius_bounds),
])
sample_specs


,dataset_id,participant_key,trial_id,label,n_samples,segment_method,train_window_count,eval_window_count
0,voisard_2025,voisard_2025:HS_1,HS_1_1,healthy,1987,event_bounds,2,2
1,felius_2024,felius_2024:H10,H10_Healthy_TestRetest_Hertest_Yes,healthy,13217,full_trial_no_event_annotation,51,26


In [8]:
split_path = PROJECT_ROOT / "data" / "interim" / "participant_splits.csv"
cross_path = PROJECT_ROOT / "data" / "interim" / "cross_dataset_splits.csv"
window_spec_path = PROJECT_ROOT / "data" / "interim" / "windowing_spec.json"

participant_splits.to_csv(split_path, index=False)
cross_dataset_splits.to_csv(cross_path, index=False)

windowing_spec = {
    "target_sampling_rate_hz": TARGET_FS_HZ,
    "window_seconds": WINDOW_SECONDS,
    "window_samples": WINDOW_SAMPLES,
    "training_hop_samples": TRAIN_HOP_SAMPLES,
    "evaluation_hop_samples": EVAL_HOP_SAMPLES,
    "channel_order": CHANNEL_ORDER,
    "voisard_segmentation": "annotated gait-event bounds with u-turn excluded",
    "felius_segmentation": "full trial in this first pass, flagged for walking-activity validation",
}
window_spec_path.write_text(json.dumps(windowing_spec, indent=2), encoding="utf-8")

print("Wrote:")
print(split_path)
print(cross_path)
print(window_spec_path)


Wrote:
C:\Users\frank\Documents\MR-ICT Review Paper\data\interim\participant_splits.csv
C:\Users\frank\Documents\MR-ICT Review Paper\data\interim\cross_dataset_splits.csv
C:\Users\frank\Documents\MR-ICT Review Paper\data\interim\windowing_spec.json


## Gate before materializing all windows

The sample loaders and split logic must be reviewed before generating large processed arrays.

Open questions that remain explicit:

1. Felius needs a validated walking-activity detector. Its first-pass full-trial segmentation is a temporary, flagged condition.
2. The common raw input uses 18 channels: three placements multiplied by three accelerometer and three gyroscope axes.
3. Participant-level splits are created before any window generation.
4. Cross-dataset testing is represented separately from internal development folds.
5. The next notebook should materialize windows only after this preprocessing contract is accepted.
